In [41]:
import pandas as pd
import pickle
import re
import ast
from tqdm import tqdm
tqdm.pandas()

import random
from random import shuffle
random.seed(42)

import torch
from transformers import AutoTokenizer, AutoModel

In [42]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

In [43]:
root_dir = '/home/DAHS1/gangmin/my_research/'

In [44]:
# pd.set_option('display.max_rows', None)
# pd.reset_option('display.max_rows')

In [45]:
imputed_text_df = pd.read_feather(root_dir+'/processed/imputed_text_df.ftr')
imputed_text_df = imputed_text_df.sort_values(['stay_id', 'hour_slot']).reset_index(drop=True)
imputed_text_df = imputed_text_df.drop(columns=['hadm_id', 'slot_start', 'slot_end', 'was_missing', 'rad_time'])
imputed_text_df.head()

,stay_id,hour_slot,extracted_text,text_flag
0,30000646,0,None,0
1,30000646,1,None,0
2,30000646,2,None,0
3,30000646,3,None,0
4,30000646,4,Cardiac silhouette has increased in size compa...,1


In [46]:
imputed_text_df[(imputed_text_df['extracted_text'].isnull()) & (imputed_text_df['text_flag']==1)]

,stay_id,hour_slot,extracted_text,text_flag


In [47]:
test = imputed_text_df.copy()

In [48]:
def preprocess_text_for_bioclinicalbert(text: str) -> str:
    if not isinstance(text, str):
        return ""
    
    # 익명화 토큰
    text = re.sub(r"\[\*\*.*?\*\*\]", "[PHI]", text)
    text = re.sub(r"_{2,}", " ", text)
    
    # 줄바꿈 / 공백 정리
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    # 너무 짧은 텍스트 제거
    if len(text.split()) < 3:
        return ""

    return text

In [49]:
test[test['text_flag']==1]

,stay_id,hour_slot,extracted_text,text_flag
4,30000646,4,Cardiac silhouette has increased in size compa...,1
30,30000646,30,The cardiac silhouette remains enlarged probab...,1
53,30000646,53,"In comparison with the study of ___, there is ...",1
59,30000646,59,No evidence of DVT. Thrombosis of the right ...,1
80,30000646,80,As compared to prior chest radiograph from ___...,1
...,...,...,...,...
1618690,39998622,135,Compared to the prior study there is no signif...,1
1618695,39998622,140,The imaged portions of the thyroid gland are n...,1
1618710,39998622,155,Lung volumes are low and there is increased vo...,1
1618734,39998622,179,"As compared to ___ radiograph, bibasilar atel...",1


In [50]:
test['extracted_text'] = test['extracted_text'].apply(preprocess_text_for_bioclinicalbert)

test['extracted_text'] = test['extracted_text'].apply(
    lambda x: None if pd.isna(x) or str(x).strip() == '' else x
)

mask_notna = test['extracted_text'].notna()

min_indices = (
    test[mask_notna]
    .sort_values(['stay_id', 'extracted_text', 'hour_slot'])
    .drop_duplicates(subset=['stay_id', 'extracted_text'], keep='first')
    .index
)

test.loc[test['extracted_text'].notna() & ~test.index.isin(min_indices), 'extracted_text'] = None
test['text_flag'] = test['extracted_text'].notna().astype(int) # New

In [51]:
# 잘못된 버전, flag가 extracted_text를 못 따라감
test[(test['extracted_text'].isnull()) & (test['text_flag']==1)]

,stay_id,hour_slot,extracted_text,text_flag


In [52]:
# 확인 결과 정상 수정됨.
test[(test['extracted_text'].isnull()) & (test['text_flag']==1)]

,stay_id,hour_slot,extracted_text,text_flag


In [53]:
print(test['extracted_text'].notna().sum())
print(test['extracted_text'].nunique())

79465
78516


In [54]:
text_df = test.copy()

In [55]:
temp_df = text_df[text_df['extracted_text'].notnull()]
temp_df = temp_df[temp_df['hour_slot'] < 120]
baseline_text_df = temp_df.drop_duplicates(subset=['extracted_text']).reset_index(drop=True)

In [ ]:
# For baseline text model single experiment
text_df.to_feather('/home/DAHS1/gangmin/my_research/processed/baseline_text_df.ftr')

In [15]:
def tokenize_text(text):
    if not isinstance(text, str):
        return {"input_ids": [], "attention_mask": []}
        # return None

    tokenized = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors='pt'
    )
    return {
        "input_ids": tokenized['input_ids'].squeeze(0).tolist(),
        "attention_mask": tokenized['attention_mask'].squeeze(0).tolist()
    }

text_df['tokenized_text'] = text_df['extracted_text'].progress_apply(tokenize_text)

100%|██████████| 1618771/1618771 [00:36<00:00, 44223.45it/s]


In [16]:
text_df[text_df['text_flag']==1]

,stay_id,hour_slot,extracted_text,text_flag,tokenized_text
4,30000646,4,Cardiac silhouette has increased in size compa...,1,"{'input_ids': [101, 17688, 27316, 1144, 2569, ..."
30,30000646,30,The cardiac silhouette remains enlarged probab...,1,"{'input_ids': [101, 1103, 17688, 27316, 2606, ..."
53,30000646,53,"In comparison with the study of , there is lit...",1,"{'input_ids': [101, 1107, 7577, 1114, 1103, 20..."
59,30000646,59,No evidence of DVT. Thrombosis of the right ba...,1,"{'input_ids': [101, 1185, 2554, 1104, 173, 196..."
80,30000646,80,"As compared to prior chest radiograph from , t...",1,"{'input_ids': [101, 1112, 3402, 1106, 2988, 22..."
...,...,...,...,...,...
1618686,39998622,131,Compared to the prior study there is no signif...,1,"{'input_ids': [101, 3402, 1106, 1103, 2988, 20..."
1618695,39998622,140,The imaged portions of the thyroid gland are n...,1,"{'input_ids': [101, 1103, 3077, 1181, 8924, 11..."
1618710,39998622,155,Lung volumes are low and there is increased vo...,1,"{'input_ids': [101, 13093, 6357, 1132, 1822, 1..."
1618734,39998622,179,"As compared to radiograph, bibasilar atelectas...",1,"{'input_ids': [101, 1112, 3402, 1106, 2070, 15..."


In [17]:
print(text_df.loc[30, 'tokenized_text'])

{'input_ids': [101, 1103, 17688, 27316, 2606, 12089, 1930, 1496, 1106, 170, 4612, 1104, 3621, 2660, 3263, 6997, 1183, 1105, 1227, 1679, 4578, 16936, 1348, 174, 3101, 17268, 119, 5855, 4638, 2083, 16201, 1158, 1103, 1762, 1336, 7977, 170, 1679, 4578, 16936, 1348, 13734, 1105, 2691, 1107, 16684, 1700, 119, 4267, 3101, 5613, 6506, 1747, 118, 2525, 11769, 7409, 4233, 1105, 1231, 2941, 6856, 1116, 1138, 4725, 117, 1256, 3525, 1111, 2569, 13093, 6357, 1105, 1336, 1129, 2272, 1106, 9248, 26600, 5048, 14494, 119, 1649, 117, 1175, 1110, 170, 6552, 1104, 9455, 2050, 17030, 1348, 11769, 7409, 4233, 1134, 1930, 4248, 181, 25698, 4993, 26320, 1610, 16430, 7903, 26156, 119, 12104, 13093, 9505, 6146, 15574, 1168, 9505, 2272, 1106, 1227, 13093, 4182, 119, 10311, 26600, 15242, 1643, 20519, 3653, 1110, 1618, 14758, 1113, 2793, 172, 1204, 16374, 1121, 119, 14255, 2087, 19224, 2227, 11769, 7409, 4233, 1120, 1103, 1286, 13093, 2259, 1105, 1231, 8005, 10542, 12571, 1805, 2845, 2569, 1133, 1132, 2846, 1106, 

In [18]:
final_text_df = text_df[text_df['hour_slot'] <= 119].reset_index(drop=True)
final_text_df

,stay_id,hour_slot,extracted_text,text_flag,tokenized_text
0,30000646,0,None,0,"{'input_ids': [], 'attention_mask': []}"
1,30000646,1,None,0,"{'input_ids': [], 'attention_mask': []}"
2,30000646,2,None,0,"{'input_ids': [], 'attention_mask': []}"
3,30000646,3,None,0,"{'input_ids': [], 'attention_mask': []}"
4,30000646,4,Cardiac silhouette has increased in size compa...,1,"{'input_ids': [101, 17688, 27316, 1144, 2569, ..."
...,...,...,...,...,...
1039537,39998622,115,None,0,"{'input_ids': [], 'attention_mask': []}"
1039538,39998622,116,None,0,"{'input_ids': [], 'attention_mask': []}"
1039539,39998622,117,None,0,"{'input_ids': [], 'attention_mask': []}"
1039540,39998622,118,None,0,"{'input_ids': [], 'attention_mask': []}"


In [19]:
final_text_df = final_text_df.drop(columns=['extracted_text'])

In [20]:
final_text_df

,stay_id,hour_slot,text_flag,tokenized_text
0,30000646,0,0,"{'input_ids': [], 'attention_mask': []}"
1,30000646,1,0,"{'input_ids': [], 'attention_mask': []}"
2,30000646,2,0,"{'input_ids': [], 'attention_mask': []}"
3,30000646,3,0,"{'input_ids': [], 'attention_mask': []}"
4,30000646,4,1,"{'input_ids': [101, 17688, 27316, 1144, 2569, ..."
...,...,...,...,...
1039537,39998622,115,0,"{'input_ids': [], 'attention_mask': []}"
1039538,39998622,116,0,"{'input_ids': [], 'attention_mask': []}"
1039539,39998622,117,0,"{'input_ids': [], 'attention_mask': []}"
1039540,39998622,118,0,"{'input_ids': [], 'attention_mask': []}"


---

In [ ]:
# old_df
check_df = pd.read_feather('/home/DAHS1/gangmin/my_research/src/test/1016_final_text_df_5days.ftr')
check_df

,stay_id,hour_slot,text_flag,parsed_token
0,30000646,0,0,"{'attention_mask': [], 'input_ids': []}"
1,30000646,1,0,"{'attention_mask': [], 'input_ids': []}"
2,30000646,2,0,"{'attention_mask': [], 'input_ids': []}"
3,30000646,3,0,"{'attention_mask': [], 'input_ids': []}"
4,30000646,4,1,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,..."
...,...,...,...,...
1039537,39998622,115,0,"{'attention_mask': [], 'input_ids': []}"
1039538,39998622,116,0,"{'attention_mask': [], 'input_ids': []}"
1039539,39998622,117,0,"{'attention_mask': [], 'input_ids': []}"
1039540,39998622,118,0,"{'attention_mask': [], 'input_ids': []}"


In [23]:
text_df[(text_df['text_flag']==1) & (text_df['tokenized_text'].isnull())]

,stay_id,hour_slot,extracted_text,text_flag,tokenized_text
150,30002498,5,None,1,None
1519,30007565,246,None,1,None
1807,30008792,40,None,1,None
1908,30009123,4,None,1,None
1953,30009123,49,None,1,None
...,...,...,...,...,...
1615898,39981921,19,None,1,None
1616145,39982700,26,None,1,None
1618165,39992578,21,None,1,None
1618553,39997955,113,None,1,None
